# Lab 3: Pipeline NLP en español — Wikipedia ES

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabloos/nlp2026/blob/main/notebooks/lab3_nlp_pipeline_noticias_es.ipynb)

> **Antes de analizar:** lee el corpus completo en la sección 2. Son 27 textos cortos — conocerlos es clave para interpretar los resultados.

In [ ]:
from IPython.display import display, HTML

STEPS = [
    ("#3498db", "🏷️", "POS Tagging",  "tokens · lemas<br>NOUN · VERB"),
    ("#e67e22", "🔎", "NER",           "PER · LOC<br>ORG · MISC"),
    ("#9b59b6", "🔗", "Dependencias",  "sujeto<br>verbo · objeto"),
    ("#2ecc71", "📊", "LDA",           "K=3 tópicos<br>por documento"),
    ("#2c3e50", "🔀", "NER × LDA",     "entidades<br>por tópico"),
]

boxes = ""
for i, (color, icon, title, sub) in enumerate(STEPS):
    radius = "10px 0 0 10px" if i == 0 else ("0 10px 10px 0" if i == len(STEPS)-1 else "0")
    boxes += (f'<div style="background:{color}; color:white; padding:14px 16px; '
              f'border-radius:{radius}; text-align:center; min-width:110px;">'
              f'<div style="font-size:1.5em; margin-bottom:4px;">{icon}</div>'
              f'<div style="font-weight:bold; font-size:0.92em;">{title}</div>'
              f'<div style="font-size:0.75em; opacity:0.85; margin-top:4px;">{sub}</div>'
              f'</div>')
    if i < len(STEPS) - 1:
        boxes += ('<div style="display:flex; align-items:center; background:#ecf0f1; '
                  'padding:0 6px; color:#7f8c8d; font-size:1.4em;">&#9658;</div>')

display(HTML(
    '<div style="font-family:sans-serif; margin:20px 0;">'
    '<div style="display:flex; align-items:stretch; justify-content:center;">'
    + boxes +
    '</div></div>'
))

In [1]:
#@title 🔧 Setup — instalación, imports y corpus { display-mode: "form" }

# ── Instalación ──────────────────────────────────────────────────────────────
!pip install -q spacy gensim
!python -m spacy download es_core_news_md -q

# ── Imports ───────────────────────────────────────────────────────────────────
import spacy, gensim, random
import numpy as np
import pandas as pd
from gensim import corpora
from gensim.models import LdaModel
from collections import defaultdict
from IPython.display import HTML

random.seed(42); np.random.seed(42)
nlp = spacy.load('es_core_news_md')

# ── Corpus: 27 resúmenes de Wikipedia ES (9 por tópico) ──────────────────────
# Fuente: es.wikipedia.org/api/rest_v1/page/summary/<título>
CORPUS = {
    'deportes': [
        {'title': 'Real Madrid CF', 'summary': 'El Real Madrid Club de Fútbol o simplemente Real Madrid, es una entidad polideportiva con sede en Madrid, España. Fue registrada oficialmente como club de fútbol por sus socios el 6 de marzo de 1902 con el objeto de la práctica y desarrollo de este deporte. Decano del fútbol madrileño, tuvo a Julián Palacios y los hermanos Juan Padrós y Carlos Padrós como principales valedores de su creación.'},
        {'title': 'Copa Mundial de Fútbol', 'summary': 'La Copa Mundial de la FIFA, también conocida como Copa Mundial de Fútbol, Copa del Mundo o simplemente Mundial es el principal torneo internacional oficial de fútbol masculino a nivel de selecciones nacionales en el mundo.'},
        {'title': 'Rafael Nadal', 'summary': 'Rafael Nadal Parera es un extenista español. Es ampliamente considerado como el mejor tenista de la historia en pistas de tierra batida y uno de los mejores de todos los tiempos. Es el segundo tenista masculino con mayor número de títulos de Grand Slam en individuales, con 22, solo superado por el serbio Novak Djokovic, con 24. Consiguió Roland Garros en catorce ediciones, Wimbledon en dos y el Abierto de Estados Unidos en cuatro ocasiones.'},
        {'title': 'Juegos Olímpicos', 'summary': 'Los Juegos Olímpicos, Olimpiadas u Olimpíadas son el mayor evento deportivo internacional multidisciplinario en el que participan atletas de más de doscientos países de todo el mundo. Existen dos tipos: los Juegos Olímpicos de Verano y los Juegos Olímpicos de Invierno, que se realizan con un intervalo de dos años según la Carta Olímpica.'},
        {'title': 'NBA', 'summary': 'La National Basketball Association (NBA) es una liga privada de baloncesto profesional que se disputa en Estados Unidos desde 1949, cuando se fusionaron la National Basketball League y la Basketball Association of America. Los jugadores de la NBA están autorizados a competir internacionalmente tras un acuerdo especial firmado en 1989 entre la Federación Internacional de Baloncesto (FIBA) y la propia NBA.'},
        {'title': 'Fórmula 1', 'summary': 'El Campeonato Mundial de Fórmula 1 de la FIA es la principal competición de automovilismo internacional y el campeonato de deportes de motor más popular y prestigioso del mundo. La entidad que la dirige es la Federación Internacional del Automóvil (FIA). Desde 2016, tras la adquisición de Formula One Group, la empresa estadounidense Liberty Media es la responsable de gestionar y operar el campeonato.'},
        {'title': 'Liga de Campeones de la UEFA', 'summary': 'La Liga de Campeones de la UEFA, también conocida como Copa de Europa, es el torneo internacional oficial de fútbol más prestigioso a nivel de clubes en Europa. Organizado por la UEFA, se disputa anualmente desde la temporada 1955-56, y su final es el acontecimiento deportivo más seguido en el mundo, con una audiencia estimada entre 350 y 400 millones de espectadores.'},
        {'title': 'Atletismo', 'summary': 'El atletismo es un deporte olímpico que agrupa numerosas disciplinas. El término atletismo deriva de la palabra griega athlon que significa competencia o combate. En este conjunto de prácticas deportivas se busca superar a los contrarios en velocidad o en resistencia, ya sea en distancia o en mayor altura. Este deporte es considerado el deporte organizado más antiguo del mundo.'},
        {'title': 'Baloncesto', 'summary': 'El baloncesto, también conocido como básquetbol o básquet, es un deporte de equipo jugado entre dos equipos de cinco jugadores cada uno. El objetivo es anotar puntos introduciendo el balón por la canasta, un aro a 3,05 metros sobre la superficie de la pista. La puntuación es de dos o tres puntos dependiendo de la posición del tiro, o de uno si se trata de un tiro libre. El equipo ganador es el que obtiene el mayor número de puntos al finalizar el partido.'},
    ],
    'politica': [
        {'title': 'Unión Europea', 'summary': 'La Unión Europea (UE) es una comunidad política democrática y de derecho, constituida en régimen sui géneris de organización supranacional fundada para propiciar y acoger la integración y gobernanza en común de los Estados y las naciones de Europa. Está compuesta por veintisiete Estados europeos y fue establecida con la entrada en vigor del Tratado de Maastricht el 1 de noviembre de 1993.'},
        {'title': 'Naciones Unidas', 'summary': 'La Organización de las Naciones Unidas (ONU) es la organización intergubernamental mundial establecida mediante la firma de la Carta de las Naciones Unidas el 26 de junio de 1945 con el propósito declarado de mantener la paz y la seguridad internacionales, fomentar las relaciones amistosas entre estados y promover la cooperación internacional.'},
        {'title': 'Democracia', 'summary': 'La democracia es una forma de organización social y política que atribuye la titularidad del poder al conjunto de la ciudadanía. En sentido estricto, la democracia es un tipo de organización del Estado en el cual las decisiones colectivas son adoptadas por el pueblo mediante herramientas de participación directa o indirecta que confieren legitimidad a sus representantes.'},
        {'title': 'OTAN', 'summary': 'La Organización del Tratado del Atlántico Norte, también conocida como la Alianza Atlántica, es una alianza militar internacional que se rige por el Tratado del Atlántico Norte firmado el 4 de abril de 1949. La organización constituye un sistema de defensa colectiva, en el cual los Estados integrantes acordaron defender a cualquiera de sus miembros que sea atacado por una potencia externa.'},
        {'title': 'Pedro Sánchez', 'summary': 'Pedro Sánchez Pérez-Castejón es un político y economista español, actual presidente del Gobierno de España desde 2018. Es secretario general del Partido Socialista Obrero Español (PSOE) desde 2017, cargo que ya había desempeñado entre 2014 y 2016.'},
        {'title': 'Congreso de los Diputados', 'summary': 'El Congreso de los Diputados es la Cámara Baja de las Cortes Generales de España, que ejerce junto con el Senado el poder legislativo del Estado. El Congreso representa al pueblo español y está compuesto por un mínimo de 300 y un máximo de 400 diputados elegidos por sufragio universal, libre, igual, directo y secreto en circunscripciones provinciales.'},
        {'title': 'Derechos humanos', 'summary': 'Los derechos humanos son principios o normas morales que establecen ciertas pautas para el comportamiento humano, y a menudo se consagran como derechos legales tanto en el derecho interno como en el internacional. Estos derechos se reconocen universalmente como derechos inalienables y fundamentales que todo individuo posee por el simple hecho de ser humano, independientemente de su origen étnico, ubicación, idioma, religión o cualquier otra condición.'},
        {'title': 'Constitución española de 1978', 'summary': 'La Constitución española de 1978 es la norma suprema del ordenamiento jurídico español, a la que están sujetos todos los poderes públicos y ciudadanos de España desde su entrada en vigor el 29 de diciembre de 1978.'},
        {'title': 'Parlamento Europeo', 'summary': 'El Parlamento Europeo es la institución parlamentaria que en la Unión Europea representa directamente a los ciudadanos europeos y que junto al Consejo de la Unión ejerce el poder legislativo. Como órgano legislativo, se encarga de elegir y controlar al ejecutivo, la Comisión Europea, y de aprobar o rechazar los proyectos de ley que éste le proponga. Está compuesto por 720 diputados que representan al segundo mayor electorado democrático del mundo.'},
    ],
    'tecnologia': [
        {'title': 'Inteligencia artificial', 'summary': 'La inteligencia artificial (IA o AI) es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana para realizar tareas que normalmente requieren inteligencia humana.'},
        {'title': 'Aprendizaje automático', 'summary': 'El aprendizaje automático es el subcampo de las ciencias de la computación y una rama de la inteligencia artificial, cuyo objetivo es desarrollar técnicas que permitan que las computadoras aprendan. Se dice que un agente aprende cuando su desempeño mejora con la experiencia y mediante el uso de datos. Un computador observa datos, construye un modelo basado en esos datos y utiliza ese modelo como hipótesis acerca del mundo para resolver problemas.'},
        {'title': 'Internet', 'summary': 'Internet es un conjunto descentralizado de redes de comunicaciones interconectadas, que utilizan la familia de protocolos TCP/IP, lo cual garantiza que las redes físicas heterogéneas que la componen constituyen una red lógica única de alcance mundial. Sus orígenes se remontan a 1969, cuando se estableció la primera conexión de ordenadores, conocida como ARPANET, entre tres universidades de California.'},
        {'title': 'Robótica', 'summary': 'La robótica es la disciplina que se ocupa del diseño, operación, manufacturación, estudio y aplicación de autómatas o robots. Combina áreas como la ingeniería mecánica, eléctrica, electrónica, biomédica y las ciencias de la computación para crear herramientas que puedan realizar tareas de manera eficiente, rápida y en ambientes inaccesibles para los humanos.'},
        {'title': 'Cadena de bloques', 'summary': 'Una cadena de bloques (blockchain) es una estructura de datos cuya información se agrupa en conjuntos (bloques) a los que se añade meta información relativa a otro bloque de la cadena anterior en una línea temporal. Gracias a técnicas criptográficas, la información contenida en un bloque solo puede ser editada modificando todos los bloques anteriores. Esta propiedad permite su aplicación en un entorno distribuido como base de datos pública no relacional con un histórico irrefutable.'},
        {'title': 'Red neuronal', 'summary': 'Una red neuronal artificial es un modelo matemático o computacional empleado en estadística, psicología cognitiva o inteligencia artificial, vagamente inspirado en el comportamiento observado en su homólogo biológico. Las redes neuronales artificiales aprenden de los datos ajustando los pesos de las conexiones entre neuronas para minimizar el error en las predicciones del modelo.'},
        {'title': 'Energía renovable', 'summary': 'Se denomina energía renovable a la energía que se obtiene a partir de fuentes naturales virtualmente inagotables, ya sea por la inmensa cantidad de energía que contienen, o porque son capaces de regenerarse por medios naturales. Entre las fuentes de energía renovable más utilizadas se encuentran la energía solar, la energía eólica, la energía hidroeléctrica y la biomasa.'},
        {'title': 'Nanotecnología', 'summary': 'La nanotecnología se refiere a la manipulación precisa de átomos y moléculas en el diseño, caracterización y aplicación de estructuras para la fabricación de dispositivos y sistemas complejos mediante el control de la forma a escala nanométrica. A esta escala se hacen visibles los efectos de la mecánica cuántica, lo que abre posibilidades únicas para materiales con nuevas propiedades físicas, químicas y biológicas.'},
        {'title': 'Computación cuántica', 'summary': 'La computación cuántica es un paradigma de computación distinto al de la informática clásica. Se basa en el uso de bits cuánticos (cúbits), una combinación especial de unos y ceros. Los bits clásicos pueden estar en 1 o en 0, pero solo un estado a la vez, en tanto que el cúbit puede tener los dos estados simultáneamente. Esto da lugar a nuevas puertas lógicas que hacen posibles nuevos algoritmos.'},
    ],
}

# ── Utilidades de visualización ───────────────────────────────────────────────
TOPIC_COLOR = {'deportes': '#2ecc71', 'politica': '#3498db', 'tecnologia': '#e67e22'}

def render_corpus(df):
    blocks = ['<div style="font-family:sans-serif;">']
    for topic in df['topic'].unique():
        color = TOPIC_COLOR.get(topic, '#888')
        subset = df[df['topic'] == topic].reset_index(drop=True)
        blocks.append(
            f'<h3 style="margin-top:28px; color:{color}; border-bottom:3px solid {color}; padding-bottom:6px;">'
            f'{topic.upper()} &nbsp;<span style="font-weight:normal; font-size:0.85em; color:#888;">({len(subset)} documentos)</span></h3>'
            f'<div style="display:grid; grid-template-columns:repeat(3,1fr); gap:12px; margin-top:10px;">'
        )
        for i, row in subset.iterrows():
            doc_idx = df[df['topic'] == topic].index[i]
            blocks.append(
                f'<div style="border-radius:8px; overflow:hidden; box-shadow:0 1px 5px rgba(0,0,0,0.13); display:flex; flex-direction:column;">'
                f'<div style="background:{color}; padding:8px 12px; color:white;">'
                f'<div style="font-size:0.70em; opacity:0.85; letter-spacing:.04em;">DOC {doc_idx}</div>'
                f'<div style="font-weight:bold; font-size:0.90em; margin-top:2px; line-height:1.3;">{row["title"]}</div>'
                f'</div>'
                f'<div style="padding:10px 12px; font-size:0.82em; color:#444; background:white; line-height:1.45; flex:1;">'
                f'{row["summary"]}'
                f'</div></div>'
            )
        blocks.append('</div>')
    blocks.append('</div>')
    return HTML('\n'.join(blocks))


# ── Utilidades de visualización POS ─────────────────────────────────────────
POS_COLOR = {
    'NOUN':  ('#2980b9', 'white'),
    'PROPN': ('#1a5276', 'white'),
    'VERB':  ('#27ae60', 'white'),
    'ADJ':   ('#d35400', 'white'),
    'ADV':   ('#7d3c98', 'white'),
    'PUNCT': ('#d5d8dc', '#666'),
    'DET':   ('#eaecee', '#888'),
    'ADP':   ('#eaecee', '#888'),
}
DEFAULT_COLOR = ('#f2f3f4', '#999')

def render_pos_doc(doc_idx, spacy_doc, topic, accent, keep_pos):
    chips = []
    for token in spacy_doc:
        bg, fg = POS_COLOR.get(token.pos_, DEFAULT_COLOR)
        kept = token.pos_ in keep_pos and not token.is_stop and token.is_alpha and len(token) > 2
        outline = f'box-shadow:0 0 0 2px {accent};' if kept else ''
        chips.append(
            f'<span style="display:inline-block; margin:2px 1px; padding:2px 6px;'
            f'background:{bg}; color:{fg}; border-radius:3px; font-size:0.82em; {outline}">'
            f'{token.text}</span>'
        )
    return (
        f'<div style="margin:14px 0 4px; font-family:sans-serif;">'
        f'<span style="font-weight:bold; color:{accent}; font-size:0.88em;">DOC {doc_idx} · {topic.upper()}</span>'
        f'<span style="font-size:0.78em; color:#aaa; margin-left:8px;">(borde coloreado = entra en LDA)</span>'
        f'</div>'
        f'<div style="line-height:1.9; font-family:sans-serif;">{"".join(chips)}</div>'
    )

POS_LEGEND = ''.join(
    f'<span style="display:inline-block; background:{c}; color:{"white" if c != "#d5d8dc" else "#666"}; '
    f'padding:2px 8px; border-radius:3px; font-size:0.78em; margin:2px;">{tag}</span>'
    for tag, c in [('NOUN','#2980b9'),('PROPN','#1a5276'),('VERB','#27ae60'),
                   ('ADJ','#d35400'),('ADV','#7d3c98'),('otras','#d5d8dc')]
)


# ── Procesado POS ────────────────────────────────────────────────────────────
def compute_pos(docs, df_corpus, keep_pos):
    pos_rows, filtered_tokens = [], []
    for i, doc in enumerate(docs):
        kept = []
        for token in doc:
            pos_rows.append({'doc': i, 'topic': df_corpus.loc[i, 'topic'],
                             'token': token.text, 'lema': token.lemma_, 'POS': token.pos_})
            if token.pos_ in keep_pos and not token.is_stop and token.is_alpha and len(token) > 2:
                kept.append(token.lemma_.lower())
        filtered_tokens.append(kept)
    return pos_rows, filtered_tokens

def display_pos(docs, df_corpus, filtered_tokens, keep_pos):
    html = [f'<div style="font-family:sans-serif;"><div style="margin-bottom:8px;">{POS_LEGEND}</div>']
    shown = set()
    for i, row in df_corpus.iterrows():
        topic = row['topic']
        if topic not in shown:
            html.append(render_pos_doc(i, docs[i], topic, TOPIC_COLOR.get(topic, '#888'), keep_pos))
            shown.add(topic)
        if len(shown) == 3:
            break
    html.append('</div>')
    display(HTML('\n'.join(html)))
    avg = np.mean([len(t) for t in filtered_tokens])
    print(f'Tokens filtrados por doc (promedio): {avg:.1f}')


# ── Utilidades NER ───────────────────────────────────────────────────────────
LABEL_MAP = {'PER': 'Persona', 'LOC': 'Lugar', 'ORG': 'Organización', 'MISC': 'Misc'}
NER_COLOR  = {'PER': '#e74c3c', 'ORG': '#2980b9', 'LOC': '#27ae60', 'MISC': '#95a5a6'}

NER_LEGEND = ''.join(
    f'<span style="display:inline-block; background:{NER_COLOR[l]}; color:white; '
    f'padding:2px 8px; border-radius:3px; font-size:0.78em; margin:2px;">{l} · {LABEL_MAP[l]}</span>'
    for l in ['PER', 'ORG', 'LOC', 'MISC']
)

def compute_ner(docs, df_corpus, keep_labels):
    rows = []
    for i, doc in enumerate(docs):
        for ent in doc.ents:
            if ent.label_ in keep_labels:
                rows.append({'doc': i, 'topic': df_corpus.loc[i, 'topic'],
                             'entidad': ent.text,
                             'tipo': LABEL_MAP.get(ent.label_, ent.label_),
                             'etiqueta': ent.label_})
    return pd.DataFrame(rows)

def render_ner_doc(doc_idx, spacy_doc, topic, accent, keep_labels):
    parts, last = [], 0
    for ent in spacy_doc.ents:
        if ent.label_ not in keep_labels:
            continue
        parts.append(spacy_doc.text[last:ent.start_char])
        color = NER_COLOR.get(ent.label_, "#95a5a6")
        parts.append(
            f'<mark style="background:{color}; color:white; padding:1px 6px; '
            f'border-radius:3px; font-size:0.88em; font-weight:bold;">{ent.text}'
            f'<sup style="font-size:0.68em; margin-left:3px; opacity:0.9;">{ent.label_}</sup></mark>'
        )
        last = ent.end_char
    parts.append(spacy_doc.text[last:])
    return (
        f'<div style="margin:14px 0 4px; font-family:sans-serif;">'
        f'<span style="font-weight:bold; color:{accent}; font-size:0.88em;">DOC {doc_idx} · {topic.upper()}</span>'
        f'</div>'
        f'<div style="font-family:sans-serif; font-size:0.88em; line-height:1.9; color:#333;">{"".join(parts)}</div>'
    )

def display_ner(df_ner, docs, df_corpus, keep_labels):
    html = [f'<div style="font-family:sans-serif;"><div style="margin-bottom:8px;">{NER_LEGEND}</div>']
    shown = set()
    for i, row in df_corpus.iterrows():
        topic = row['topic']
        if topic not in shown:
            html.append(render_ner_doc(i, docs[i], topic, TOPIC_COLOR.get(topic, '#888'), keep_labels))
            shown.add(topic)
        if len(shown) == 3:
            break
    html.append('</div>')
    display(HTML('\n'.join(html)))
    display(HTML('<div style="margin-top:24px; border-top:1px solid #eee; padding-top:12px;"></div>'))
    print(f'Entidades encontradas: {len(df_ner)}')
    if len(df_ner):
        display(df_ner.groupby(['etiqueta','tipo']).size().reset_index(name='n').sort_values('n', ascending=False))

# ── Utilidades SVO ───────────────────────────────────────────────────────────
def compute_svo(docs, df_corpus, subj_dep, obj_dep):
    rows = []
    for i, doc in enumerate(docs):
        for token in doc:
            if token.pos_ == 'VERB':
                subjs = [c for c in token.children if c.dep_ in subj_dep]
                objs  = [c for c in token.children if c.dep_ in obj_dep]
                for s in subjs:
                    for o in objs:
                        clean = (s.pos_ in {'NOUN','PROPN'} and not s.is_stop and len(s.text) > 2
                                 and o.pos_ in {'NOUN','PROPN'} and not o.is_stop and len(o.text) > 2)
                        rows.append({'doc': i, 'topic': df_corpus.loc[i, 'topic'],
                                     'sujeto': s.text, 'verbo': token.lemma_, 'objeto': o.text,
                                     'dep_suj': s.dep_, 'dep_obj': o.dep_, 'limpia': clean})
    cols = ['doc','topic','sujeto','verbo','objeto','dep_suj','dep_obj','limpia']
    df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=cols)
    n = len(docs)
    covered = int(df['doc'].nunique()) if len(df) else 0
    metrics = {
        'total': len(df), 'docs_cubiertos': covered, 'n_docs': n,
        'cobertura': covered / n,
        'precision_aprox': float(df['limpia'].mean()) if len(df) else 0.0,
    }
    return df, metrics

def display_svo(df_svo, metrics):
    if len(df_svo) == 0:
        print('Sin tripletas con esta configuración.')
        return
    print(f'Tripletas extraídas: {metrics["total"]}  |  documentos cubiertos: {metrics["docs_cubiertos"]}/{metrics["n_docs"]}')
    clean = df_svo[df_svo['limpia']][['sujeto','verbo','objeto','dep_suj','dep_obj']].head(6)
    noisy = df_svo[~df_svo['limpia']][['sujeto','verbo','objeto','dep_suj','dep_obj']].head(6)
    def _rows(df, color):
        return ''.join(
            f'<tr><td style="padding:3px 8px; color:{color}; font-weight:bold;">{r.sujeto}</td>'
            f'<td style="padding:3px 8px;">{r.verbo}</td>'
            f'<td style="padding:3px 8px; color:{color}; font-weight:bold;">{r.objeto}</td>'
            f'<td style="padding:3px 8px; font-size:0.78em; color:#888;">{r.dep_suj}/{r.dep_obj}</td></tr>'
            for r in df.itertuples()
        )
    hdr = '<tr style="font-size:0.8em; color:#888; border-bottom:1px solid #ddd;"><th style="padding:3px 8px; text-align:left;">sujeto</th><th style="padding:3px 8px; text-align:left;">verbo</th><th style="padding:3px 8px; text-align:left;">objeto</th><th style="padding:3px 8px; text-align:left;">dep</th></tr>'
    table = (
        f'<div style="display:grid; grid-template-columns:1fr 1fr; gap:16px; font-family:sans-serif; font-size:0.88em; margin-top:16px;">'
        f'<div><div style="font-weight:bold; color:#27ae60; margin-bottom:6px;">✓ Limpias — suj+obj son sustantivos</div>'
        f'<table style="border-collapse:collapse; width:100%;">{hdr}{_rows(clean, "#27ae60")}</table></div>'
        f'<div><div style="font-weight:bold; color:#e67e22; margin-bottom:6px;">⚠ Ruidosas — pronombres o palabras vacías</div>'
        f'<table style="border-collapse:collapse; width:100%;">{hdr}{_rows(noisy, "#e67e22")}</table></div>'
        f'</div>'
    )
    display(HTML(table))


# ── Utilidades LDA ────────────────────────────────────────────────────────────
def compute_lda(filtered_tokens, K, alpha, eta):
    dictionary = corpora.Dictionary(filtered_tokens)
    dictionary.filter_extremes(no_below=2, no_above=0.9)
    bow = [dictionary.doc2bow(t) for t in filtered_tokens]
    lda = LdaModel(bow, id2word=dictionary, num_topics=K,
                   random_state=42, passes=40, alpha=alpha, eta=eta)
    dist = [dict(lda.get_document_topics(b, minimum_probability=0.0)) for b in bow]
    return lda, bow, dist

def display_lda(lda, doc_topic_dist, df_corpus, K):
    concentracion = np.mean([max(d.values()) for d in doc_topic_dist])
    color = '#27ae60' if concentracion >= 0.7 else '#e67e22' if concentracion >= 0.5 else '#e74c3c'
    panel = (
        f'<div style="font-family:sans-serif; display:flex; gap:16px; margin:14px 0;">'
        f'<div style="background:#f8f9fa; border-radius:8px; padding:14px 22px; text-align:center;">'
        f'<div style="font-size:1.9em; font-weight:bold; color:{color}">{concentracion:.0%}</div>'
        f'<div style="font-size:0.8em; color:#666; margin-top:4px;">concentración<br>'
        f'<span style="opacity:0.7; font-size:0.9em;">'
        f'peso medio del tópico dominante por doc<br>'
        f'&gt;70% = bien separados · &lt;50% = solapamiento excesivo'
        f'</span></div></div></div>'
    )
    display(HTML(panel))
    rows = []
    for t in range(K):
        for rank, (word, _) in enumerate(lda.show_topic(t, topn=5), 1):
            rows.append({'tópico': f'T{t}', 'rank': rank, 'palabra': word})
    print('Top-5 palabras por tópico:')
    display(pd.DataFrame(rows).pivot(index='rank', columns='tópico', values='palabra'))
    mix = []
    for i, dist in enumerate(doc_topic_dist):
        row = {'doc': i, 'topic_real': df_corpus.loc[i, 'topic']}
        for t in range(K):
            row[f'T{t}'] = round(dist.get(t, 0.0), 3)
        row['T_dom'] = f'T{max(dist, key=dist.get)}'
        mix.append(row)
    print()
    print('Distribución de tópicos por documento:')
    display(pd.DataFrame(mix))


n_docs = sum(len(v) for v in CORPUS.values())
print(f'✓ spaCy {spacy.__version__} | Gensim {gensim.__version__} | {n_docs} documentos listos')

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
✓ spaCy 3.8.11 | Gensim 4.4.0 | 27 documentos listos


## 2. Corpus

**27 resúmenes de Wikipedia en español** — 9 artículos por tópico (deportes, política, tecnología).

Lee los textos antes de continuar: son cortos y conocerlos es clave para interpretar los resultados.

In [2]:
# Celda 2: Construir el corpus, mostrarlo y procesarlo con spaCy
df_corpus = pd.DataFrame([
    {'topic': topic, 'title': art['title'], 'summary': art['summary']}
    for topic, arts in CORPUS.items()
    for art in arts
]).reset_index(drop=True)

display(render_corpus(df_corpus))

docs = list(nlp.pipe(df_corpus['summary'].tolist()))
avg_words = round(sum(len([t for t in d if t.is_alpha]) for d in docs) / len(docs))
print(f'{len(docs)} documentos procesados con spaCy | promedio de palabras por doc: {avg_words}')

27 documentos procesados con spaCy | promedio de palabras por doc: 59


## 3. POS Tagging — Etiquetado gramatical

Cada token recibe una etiqueta morfológica. Los tokens con **borde coloreado** son los que pasarán al modelo LDA.

---

**✏️ Ejercicio:** `KEEP_POS` controla qué categorías entran en LDA. La celda incluye todas las candidatas relevantes — **borra las que no quieras conservar** y vuelve a ejecutar para ver cómo cambia la visualización y, más adelante, los tópicos.

> Pista: ¿aportan los adjetivos información temática? ¿y los adverbios?

In [3]:
# ✏️ Borra las categorías que no quieras conservar y vuelve a ejecutar
KEEP_POS = {
    'NOUN',   # sustantivo común  → "fútbol", "organización", "algoritmo"
    'PROPN',  # sustantivo propio → "Madrid", "UEFA", "Internet"
    'VERB',   # verbo             → "ganar", "establecer", "computar"
    'ADJ',    # adjetivo          → "internacional", "cuántico", "democrático"
    'ADV',    # adverbio          → "ampliamente", "oficialmente"
}

pos_rows, filtered_tokens = compute_pos(docs, df_corpus, KEEP_POS)
display_pos(docs, df_corpus, filtered_tokens, KEEP_POS)

Tokens filtrados por doc (promedio): 28.2


## 4. NER — Reconocimiento de Entidades Nombradas

Cada span del texto recibe una etiqueta de entidad. Las entidades seleccionadas alimentarán el cruce NER × LDA de la sección 7.

---

**✏️ Ejercicio:** `KEEP_LABELS` controla qué tipos de entidad se conservan — **borra los que no quieras** y observa cómo cambia el resultado del cruce final.

> Pista: ¿añade `MISC` señal real o solo ruido? ¿es `LOC` igual de discriminativo que `PER` u `ORG`?

In [4]:
# ✏️ Borra los tipos que no quieras conservar y vuelve a ejecutar esta celda
KEEP_LABELS = {
    'PER',   # personas        → Rafael Nadal, Pedro Sánchez
    'ORG',   # organizaciones  → UEFA, ONU, NBA
    'LOC',   # lugares         → España, Europa, California
    'MISC',  # miscelánea      → suele ser ruidoso
}

df_ner = compute_ner(docs, df_corpus, KEEP_LABELS)
display_ner(df_ner, docs, df_corpus, KEEP_LABELS)

Entidades encontradas: 94


,etiqueta,tipo,n
1,MISC,Misc,39
2,ORG,Organización,30
0,LOC,Lugar,14
3,PER,Persona,11


## 5. Análisis de Dependencias — Tripletas Sujeto-Verbo-Objeto

El parser de dependencias de spaCy asigna a cada token un **arco** que lo conecta a su cabeza sintáctica. Recorremos esos arcos para extraer tripletas *(sujeto, verbo, objeto)*.

### ¿Qué cuenta como "objeto"?

| Etiqueta | Qué captura | Ejemplo |
|---|---|---|
| `obj` | objeto directo | *"ganó **el campeonato**"*, *"constituir **un sistema**"* |
| `obl` | oblicuo — complemento preposicional | *"disputar **en Estados Unidos**"*, *"fundada **en 1902**"* |

`obl` no son objetos semánticos reales — son complementos de lugar, tiempo o modo. Incluirlos aumenta el número de tripletas extraídas pero introduce ruido.

---

**✏️ Ejercicio:** descomenta `'obl'` y vuelve a ejecutar.

- ¿Cuántas tripletas nuevas aparecen?
- Mira la columna **ruidosas** — ¿las nuevas tripletas tienen sentido semántico?
- ¿Vale la pena incluir `obl` para este corpus?

In [5]:
# ✏️ Prueba las dos configuraciones del ejercicio y compara las métricas

SUBJ_DEP = {'nsubj'}   # sujeto activo

OBJ_DEP = {
    'obj',   # objeto directo  → más limpio
     #'obl', # oblicuo         → más cobertura, ¿a qué coste? (descomenta para probar)
}

df_svo, metrics = compute_svo(docs, df_corpus, SUBJ_DEP, OBJ_DEP)
display_svo(df_svo, metrics)

Tripletas extraídas: 35  |  documentos cubiertos: 16/27


sujeto,verbo,objeto,dep
Decano,tener,Julián,nsubj/obj
deporte,considerar,deporte,nsubj/obj
decisiones,adoptar,pueblo,nsubj/obj
organización,constituir,sistema,nsubj/obj
Congreso,representar,pueblo,nsubj/obj
computador,observar,datos,nsubj/obj
sujeto,verbo,objeto,dep
que,dirigir,la,nsubj/obj
Federación,dirigir,la,nsubj/obj
que,agrupar,disciplinas,nsubj/obj


## 6. LDA — Modelado de Tópicos

**Latent Dirichlet Allocation (LDA)** asume que cada documento es una mezcla de tópicos latentes y cada tópico es una distribución sobre palabras.

**K** es el hiperparámetro más crítico: determina cuántos tópicos busca el modelo. No hay un valor universalmente correcto — depende de la granularidad que queramos y del corpus.

La **concentración** mide el peso medio del tópico dominante por documento: un indicador rápido de si el modelo separa bien o todo se solapa.

---

**✏️ Ejercicio:** cambia `K` y observa:
- ¿Qué pasa con la concentración?
- ¿Son los tópicos más o menos interpretables?
- ¿A partir de qué K empiezan a aparecer tópicos redundantes o sin sentido?

In [6]:
# Celda 6: LDA
# ✏️ Cambia K y observa cómo cambian los tópicos y la concentración

K = 3   # número de tópicos — prueba K=2, K=4, K=5

lda, bow_corpus, doc_topic_dist = compute_lda(filtered_tokens, K, alpha='auto', eta='auto')
display_lda(lda, doc_topic_dist, df_corpus, K)

Top-5 palabras por tópico:


tópico,T0,T1,T2
rank,,,
1,español,internacional,inteligencia
2,derecho,mundial,artificial
3,deporte,red,organización
4,españa,mundo,internacional
5,unión,conocido,modelo



Distribución de tópicos por documento:


,doc,topic_real,T0,T1,T2,T_dom
0,0,deportes,0.993,0.003,0.004,T0
1,1,deportes,0.002,0.995,0.002,T1
2,2,deportes,0.983,0.008,0.009,T0
3,3,deportes,0.008,0.008,0.984,T2
4,4,deportes,0.004,0.004,0.992,T2
5,5,deportes,0.004,0.991,0.005,T1
6,6,deportes,0.003,0.995,0.003,T1
7,7,deportes,0.992,0.004,0.004,T0
8,8,deportes,0.004,0.004,0.992,T2
9,9,politica,0.994,0.003,0.003,T0


### Nombrar los tópicos

LDA descubre distribuciones sobre palabras — no etiquetas. En un corpus **no etiquetado** este paso lo hace siempre un humano: leer las top-5 palabras de cada tópico e interpretarlas.

> Con nuestro corpus ya sabemos las categorías reales (deportes, política, tecnología).
> Pero imagina que no las supieras: ¿qué nombre le pondrías a cada tópico solo mirando las palabras de arriba?

**✏️ Ejercicio:** rellenas `TOPIC_NAMES` con el nombre que le darías a cada tópico y vuelve a ejecutar la sección 7.

In [7]:
# ✏️ Asigna un nombre a cada tópico según las palabras que viste arriba
TOPIC_NAMES = {f'T{t}': '???' for t in range(K)}

# Ejemplo una vez que lo hayas rellenado:
# TOPIC_NAMES = {'T0': 'Política', 'T1': 'Tecnología', 'T2': 'Deportes'}

print('Tópicos con sus nombres actuales:')
for tid, name in TOPIC_NAMES.items():
    words = [w for w, _ in lda.show_topic(int(tid[1]), topn=5)]
    print(f'  {tid} → {name:<20} palabras: {words}')

Tópicos con sus nombres actuales:
  T0 → ???                  palabras: ['español', 'derecho', 'deporte', 'españa', 'unión']
  T1 → ???                  palabras: ['internacional', 'mundial', 'red', 'mundo', 'conocido']
  T2 → ???                  palabras: ['inteligencia', 'artificial', 'organización', 'internacional', 'modelo']


## 7. Conclusión — Cruce NER × LDA

Esta sección es el resultado acumulado de todas las decisiones anteriores. Antes de ejecutar, revisa la cadena:

| Sección | Parámetro | Efecto aquí |
|---|---|---|
| 3 · POS | `KEEP_POS` | define el vocabulario de LDA → qué palabras forman los tópicos |
| 4 · NER | `KEEP_LABELS` | qué tipos de entidad entran al cruce |
| 6 · LDA | `K` | cuántos tópicos hay para clasificar las entidades |

Si modificaste algún parámetro en su sección, **vuelve a ejecutar las celdas en orden** y luego esta para ver el efecto propagado.

In [8]:
# Celda 7: Cruce NER × LDA

# ── Trazabilidad: qué llegó hasta aquí ───────────────────────────────────────
avg_tok = np.mean([len(t) for t in filtered_tokens])
print(f"KEEP_POS    → {sorted(KEEP_POS)}")
print(f"             {avg_tok:.0f} tokens/doc · vocabulario LDA: {len(lda.id2word)} términos")
print(f"KEEP_LABELS → {sorted(KEEP_LABELS)}")
print(f"             {len(df_ner)} entidades ({df_ner['etiqueta'].value_counts().to_dict()})")
print(f"K           → {K} tópicos:")
for t in range(K):
    words = [w for w, _ in lda.show_topic(t, topn=5)]
    name  = TOPIC_NAMES.get(f'T{t}', f'T{t}')
    print(f"             T{t} '{name}': {words}")
print()

# ── Cruce ─────────────────────────────────────────────────────────────────────
entity_acc = defaultdict(lambda: defaultdict(float))
for _, row in df_ner.iterrows():
    label = f"{row['entidad']} ({row['tipo']})"
    dist  = doc_topic_dist[row['doc']]
    for t in range(K):
        entity_acc[label][f'T{t}'] += dist.get(t, 0.0)

cross_rows = []
for entity, weights in entity_acc.items():
    row = {'entidad': entity}
    row.update({k: round(v, 3) for k, v in weights.items()})
    dom = max(weights, key=weights.get)
    row['T_dom']  = dom
    row['nombre'] = TOPIC_NAMES.get(dom, dom)
    cross_rows.append(row)

df_cross = pd.DataFrame(cross_rows).sort_values('nombre').reset_index(drop=True)
print('=== Entidades NER clasificadas por tópico ===')
display(df_cross)

print()
print('=== Resumen: entidades destacadas por tópico ===')
fallback = '(ninguna dominante)'
for t in range(K):
    tid   = f'T{t}'
    name  = TOPIC_NAMES.get(tid, tid)
    words = [w for w, _ in lda.show_topic(t, topn=5)]
    ents  = df_cross[df_cross['T_dom'] == tid]['entidad'].tolist()
    print(f"{tid} '{name}' | palabras: {words}")
    print(f"     entidades: {ents[:8] if ents else fallback}")
    print()

KEEP_POS    → ['ADJ', 'ADV', 'NOUN', 'PROPN', 'VERB']
             28 tokens/doc · vocabulario LDA: 96 términos
KEEP_LABELS → ['LOC', 'MISC', 'ORG', 'PER']
             94 entidades ({'MISC': 39, 'ORG': 30, 'LOC': 14, 'PER': 11})
K           → 3 tópicos:
             T0 '???': ['español', 'derecho', 'deporte', 'españa', 'unión']
             T1 '???': ['internacional', 'mundial', 'red', 'mundo', 'conocido']
             T2 '???': ['inteligencia', 'artificial', 'organización', 'internacional', 'modelo']

=== Entidades NER clasificadas por tópico ===


,entidad,T0,T1,T2,T_dom,nombre
0,Real Madrid Club de Fútbol (Organización),0.993,0.003,0.004,T0,???
1,Real Madrid (Organización),0.993,0.003,0.004,T0,???
2,Madrid (Lugar),0.993,0.003,0.004,T0,???
3,España (Lugar),1.983,0.008,0.009,T0,???
4,Decano (Persona),0.993,0.003,0.004,T0,???
...,...,...,...,...,...,...
81,Gracias a técnicas criptográficas (Misc),0.004,0.992,0.004,T1,???
82,Esta propiedad (Misc),0.004,0.992,0.004,T1,???
83,Las redes neuronales artificiales aprenden de ...,0.003,0.258,0.739,T2,???
84,Entre las fuentes de energía renovable más uti...,0.036,0.036,0.928,T2,???



=== Resumen: entidades destacadas por tópico ===
T0 '???' | palabras: ['español', 'derecho', 'deporte', 'españa', 'unión']
     entidades: ['Real Madrid Club de Fútbol (Organización)', 'Real Madrid (Organización)', 'Madrid (Lugar)', 'España (Lugar)', 'Decano (Persona)', 'Julián Palacios (Persona)', 'Juan Padrós (Persona)', 'Carlos Padrós (Persona)']

T1 '???' | palabras: ['internacional', 'mundial', 'red', 'mundo', 'conocido']
     entidades: ['Copa Mundial de la FIFA (Misc)', 'Copa Mundial de Fútbol (Misc)', 'Copa del Mundo (Misc)', 'Mundial (Misc)', 'Campeonato Mundial de Fórmula 1 (Misc)', 'FIA (Organización)', 'La entidad que la dirige es la (Misc)', 'Federación Internacional del Automóvil (Organización)']

T2 '???' | palabras: ['inteligencia', 'artificial', 'organización', 'internacional', 'modelo']
     entidades: ['Juegos Olímpicos (Misc)', 'Olimpiadas (Misc)', 'Olimpíadas (Misc)', 'Existen dos tipos (Misc)', 'Juegos Olímpicos de Verano (Misc)', 'Juegos Olímpicos de Invierno (M

## Notas metodológicas

| Técnica | Herramienta | Entrada | Salida |
|---|---|---|---|
| POS tagging | spaCy `es_core_news_md` | token | etiqueta POS + lema |
| NER | spaCy `es_core_news_md` | span | PER / LOC / ORG / MISC |
| Dep. parsing (SVO) | spaCy `es_core_news_md` | oración | tripletas S-V-O |
| LDA | Gensim | resumen (~80 palabras) | distribución sobre K=3 tópicos |
| Cruce NER × LDA | pandas | entidad + doc | entidades clasificadas por tópico |

**Por qué resúmenes y no frases sueltas:**  
LDA necesita suficientes palabras por documento para estimar distribuciones de tópicos con fiabilidad.
Los resúmenes de Wikipedia (~80 palabras) son legibles *y* estadísticamente manejables.

**Por qué corpus hardcodeado:**  
Los paquetes de datasets más usados (MLSUM, XL-Sum) usan scripts de carga legacy incompatibles con `datasets >= 3.x`.
El corpus se obtuvo una vez de la Wikipedia REST API y se incrustó directamente para garantizar reproducibilidad sin dependencias de red.

**Sobre el SVO en español:**  
El orden de palabras libre (VSO, OVS son frecuentes) hace que el parser recupere menos tripletas
que en inglés — es una limitación real del método, no un bug del código.

**Corpus:** 27 artículos de [Wikipedia en español](https://es.wikipedia.org) — 9 por tópico (deportes, política, tecnología).